In [ ]:
#import packages
import json
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from torch.optim import Adam
from tqdm import tqdm

In [ ]:
#define configuration
class CFG:
    seed = 42
    model_name = "microsoft/deberta-v3-small"
    max_length = 512
    batch_size = 8
    epochs = 3
    lr = 5e-6
    num_classes = 3
    label2name = {0: 'winner_model_a', 1: 'winner_model_b', 2: 'winner_tie'}
    name2label = {v: k for k, v in label2name.items()}

In [ ]:
#define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
#define base path
BASE_PATH = "data/llm-classification-finetuning"

In [10]:
#Load data and convert "null" to ""
df = pd.read_csv(f"{BASE_PATH}/train.csv")
df["prompt"] = df["prompt"].map(lambda x: json.loads(x)[0] if pd.notna(x) else "")
df["response_a"] = df["response_a"].map(lambda x: json.loads(x)[0] if pd.notna(x) else "")
df["response_b"] = df["response_b"].map(lambda x: json.loads(x)[0] if pd.notna(x) else "")

test_df = pd.read_csv(f"{BASE_PATH}/test.csv")
test_df["prompt"] = test_df["prompt"].map(lambda x: json.loads(x)[0] if pd.notna(x) else "")
test_df["response_a"] = test_df["response_a"].map(lambda x: json.loads(x)[0] if pd.notna(x) else "")
test_df["response_b"] = test_df["response_b"].map(lambda x: json.loads(x)[0] if pd.notna(x) else "")

df["class_name"] = df[["winner_model_a", "winner_model_b", "winner_tie"]].idxmax(axis=1)
df["label"] = df["class_name"].map(CFG.name2label)

In [11]:
#concat prompt and response
def build_text(prompt, response):
    return f"Prompt: {prompt}\n\nResponse: {response}"

df["text_a"] = df.apply(lambda r: build_text(r["prompt"], r["response_a"]), axis=1)
df["text_b"] = df.apply(lambda r: build_text(r["prompt"], r["response_b"]), axis=1)
test_df["text_a"] = test_df.apply(lambda r: build_text(r["prompt"], r["response_a"]), axis=1)
test_df["text_b"] = test_df.apply(lambda r: build_text(r["prompt"], r["response_b"]), axis=1)

In [12]:
display(df.head())
display(test_df.head())

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie,class_name,label,text_a,text_b
0,30192,gpt-4-1106-preview,gpt-4-0613,Is it morally right to try to have a certain p...,The question of whether it is morally right to...,"As an AI, I don't have personal beliefs or opi...",1,0,0,winner_model_a,0,Prompt: Is it morally right to try to have a c...,Prompt: Is it morally right to try to have a c...
1,53567,koala-13b,gpt-4-0613,What is the difference between marriage licens...,A marriage license is a legal document that al...,A marriage license and a marriage certificate ...,0,1,0,winner_model_b,1,Prompt: What is the difference between marriag...,Prompt: What is the difference between marriag...
2,65089,gpt-3.5-turbo-0613,mistral-medium,explain function calling. how would you call a...,Function calling is the process of invoking or...,Function calling is the process of invoking a ...,0,0,1,winner_tie,2,Prompt: explain function calling. how would yo...,Prompt: explain function calling. how would yo...
3,96401,llama-2-13b-chat,mistral-7b-instruct,How can I create a test set for a very rare ca...,Creating a test set for a very rare category c...,When building a classifier for a very rare cat...,1,0,0,winner_model_a,0,Prompt: How can I create a test set for a very...,Prompt: How can I create a test set for a very...
4,198779,koala-13b,gpt-3.5-turbo-0314,What is the best way to travel from Tel-Aviv t...,The best way to travel from Tel Aviv to Jerusa...,The best way to travel from Tel-Aviv to Jerusa...,0,1,0,winner_model_b,1,Prompt: What is the best way to travel from Te...,Prompt: What is the best way to travel from Te...


,id,prompt,response_a,response_b,text_a,text_b
0,136060,"I have three oranges today, I ate an orange ye...",You have two oranges today.,You still have three oranges. Eating an orange...,"Prompt: I have three oranges today, I ate an o...","Prompt: I have three oranges today, I ate an o..."
1,211333,You are a mediator in a heated political debat...,Thank you for sharing the details of the situa...,Mr Reddy and Ms Blue both have valid points in...,Prompt: You are a mediator in a heated politic...,Prompt: You are a mediator in a heated politic...
2,1233961,How to initialize the classification head when...,When you want to initialize the classification...,To initialize the classification head when per...,Prompt: How to initialize the classification h...,Prompt: How to initialize the classification h...


In [13]:
train_df, valid_df = train_test_split(df, test_size=0.2, random_state=CFG.seed, stratify=df["label"])

In [16]:
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)

Could not extract SentencePiece model from C:\Users\kajita\.cache\huggingface\hub\models--microsoft--deberta-v3-small\snapshots\a36c739020e01763fe789b4b85e2df55d6180012\spm.model using sentencepiece library due to 
SentencePieceExtractor requires the SentencePiece library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.
. Falling back to TikToken extractor.


ValueError: Error parsing line b'\x0e' in C:\Users\kajita\.cache\huggingface\hub\models--microsoft--deberta-v3-small\snapshots\a36c739020e01763fe789b4b85e2df55d6180012\spm.model